# 🇱🇰 XTTS v2 — Sinhala Fine-tuning

Fine-tune Coqui XTTS v2 on multi-speaker Sinhala data for **voice cloning + Sinhala TTS**.

- **Dataset:** `irudachirath/multi-speaket-tts-dataset-sinhala` (~600 cherry-picked entries)
- **Training:** 100 epochs, GPT encoder only
- **Result:** Voice cloning with 6-second audio clip, runs on CPU

**GPU:** L4 recommended (~1-2 hrs), T4 works too (~3-4 hrs)

## 1. SSH Tunnel (for remote monitoring)

In [ ]:
# Setup SSH access via cloudflared tunnel
!pip install colab_ssh -q

# Add your SSH public key
import os
os.makedirs('/root/.ssh', exist_ok=True)
with open('/root/.ssh/authorized_keys', 'w') as f:
    f.write('ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIFihGxxIUx5jFCkbFMZOIzoDfmMRKle7gvSDdafMYhSu chanclaw@ubuntu-4gb-hel1-2')

from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password='')

## 2. Install Dependencies

In [ ]:
!pip install TTS==0.22.0 datasets soundfile librosa -q
print('✅ Dependencies installed')

## 3. Download & Prepare Dataset

In [ ]:
import os, json, soundfile as sf, numpy as np, librosa
from datasets import load_dataset

os.makedirs('/content/xtts_sinhala/dataset/wavs', exist_ok=True)
os.makedirs('/content/xtts_sinhala/reference', exist_ok=True)
os.makedirs('/content/xtts_sinhala/output', exist_ok=True)

print('Loading multi-speaker Sinhala dataset...')
ds = load_dataset('irudachirath/multi-speaket-tts-dataset-sinhala', split='train')
print(f'Total: {len(ds)} entries, Columns: {ds.column_names}')

# Inspect structure
sample = ds[0]
for k, v in sample.items():
    if isinstance(v, dict):
        print(f'  {k}: dict keys={list(v.keys())}')
    elif isinstance(v, str):
        print(f'  {k}: "{v[:80]}"')
    else:
        print(f'  {k}: {type(v).__name__}')

In [ ]:
# Cherry-pick ~600 entries balanced across speakers
# Detect speaker field
speaker_field = None
for field in ['speaker_id', 'speaker', 'client_id', 'accent', 'gender']:
    if field in ds.column_names:
        speaker_field = field
        break

if speaker_field:
    speakers = {}
    for i, entry in enumerate(ds):
        spk = str(entry[speaker_field])
        if spk not in speakers:
            speakers[spk] = []
        speakers[spk].append(i)
    print(f'Found {len(speakers)} speakers via "{speaker_field}": {list(speakers.keys())[:10]}')
    
    target = 600
    per_speaker = max(50, target // len(speakers))
    selected = []
    for spk, indices in speakers.items():
        selected.extend(indices[:per_speaker])
    selected = selected[:target]
    print(f'Selected {len(selected)} entries balanced across speakers')
else:
    selected = list(range(min(600, len(ds))))
    print(f'No speaker field found. Using first {len(selected)} entries.')

# Detect text field
text_field = None
for field in ['sentence', 'text', 'transcription', 'normalized_text']:
    if field in ds.column_names:
        text_field = field
        break
print(f'Text field: {text_field}')

In [ ]:
# Process and save audio files in LJSpeech format
metadata = []
ref_saved = False
skipped = 0

for idx, i in enumerate(selected):
    entry = ds[i]
    audio = entry.get('audio', None)
    if audio is None:
        skipped += 1
        continue
    
    arr = np.array(audio['array'], dtype=np.float32)
    sr = audio['sampling_rate']
    duration = len(arr) / sr
    
    # Skip very short (<1s) or very long (>15s)
    if duration < 1.0 or duration > 15.0:
        skipped += 1
        continue
    
    # Resample to 22050Hz
    if sr != 22050:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=22050)
        sr = 22050
    
    wav_path = f'/content/xtts_sinhala/dataset/wavs/{idx:05d}.wav'
    sf.write(wav_path, arr, sr)
    
    text = entry.get(text_field, '')
    if not text or len(text.strip()) < 3:
        skipped += 1
        continue
    
    # Save first good clip as speaker reference for test sentences
    if not ref_saved and 3.0 < duration < 10.0:
        sf.write('/content/xtts_sinhala/reference/ref_speaker.wav', arr, sr)
        ref_saved = True
        print(f'✅ Saved reference speaker clip (duration: {duration:.1f}s)')
    
    metadata.append(f'{idx:05d}|{text.strip()}|{text.strip()}')
    
    if (idx + 1) % 100 == 0:
        print(f'  Processed {idx+1}/{len(selected)}')

with open('/content/xtts_sinhala/dataset/metadata.csv', 'w') as f:
    f.write('\n'.join(metadata))

print(f'\n=== Dataset Ready ===')
print(f'✅ {len(metadata)} entries saved')
print(f'⏭️ {skipped} skipped')
print(f'🎤 Reference speaker: {ref_saved}')

## 4. Fine-tune XTTS v2

In [ ]:
import os
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig
from TTS.utils.manage import ModelManager

OUT_PATH = '/content/xtts_sinhala/output'
BATCH_SIZE = 4
GRAD_ACUMM_STEPS = 63  # 4 * 63 = 252

config_dataset = BaseDatasetConfig(
    formatter='ljspeech',
    dataset_name='sinhala_tts',
    path='/content/xtts_sinhala/dataset/',
    meta_file_train='/content/xtts_sinhala/dataset/metadata.csv',
    language='si',
)

# Download XTTS v2 base model
CKPT_DIR = os.path.join(OUT_PATH, 'XTTS_v2_base')
os.makedirs(CKPT_DIR, exist_ok=True)

files = {
    'dvae.pth': 'https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/dvae.pth',
    'mel_stats.pth': 'https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/mel_stats.pth',
    'vocab.json': 'https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/vocab.json',
    'model.pth': 'https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/model.pth',
}

for fname, url in files.items():
    fpath = os.path.join(CKPT_DIR, fname)
    if not os.path.isfile(fpath):
        print(f'Downloading {fname}...')
        ModelManager._download_model_files([url], CKPT_DIR, progress_bar=True)
    else:
        print(f'✅ {fname} exists')

print('\n✅ Base model ready')

In [ ]:
# Configure and start training
DVAE_CHECKPOINT = os.path.join(CKPT_DIR, 'dvae.pth')
MEL_NORM_FILE = os.path.join(CKPT_DIR, 'mel_stats.pth')
TOKENIZER_FILE = os.path.join(CKPT_DIR, 'vocab.json')
XTTS_CHECKPOINT = os.path.join(CKPT_DIR, 'model.pth')
SPEAKER_REFERENCE = ['/content/xtts_sinhala/reference/ref_speaker.wav']

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=255995,
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

audio_config = XttsAudioConfig(
    sample_rate=22050,
    dvae_sample_rate=22050,
    output_sample_rate=24000,
)

config = GPTTrainerConfig(
    epochs=100,
    output_path=OUT_PATH,
    model_args=model_args,
    run_name='XTTS_v2_Sinhala_FT',
    project_name='XTTS_Sinhala',
    run_description='XTTS v2 fine-tuning for Sinhala with multi-speaker data',
    dashboard_logger='tensorboard',
    logger_uri=None,
    audio=audio_config,
    batch_size=BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=BATCH_SIZE,
    num_loader_workers=4,
    eval_split_max_size=50,
    print_step=25,
    plot_step=100,
    log_model_step=500,
    save_step=2000,
    save_n_checkpoints=2,
    save_checkpoints=True,
    print_eval=False,
    optimizer='AdamW',
    optimizer_wd_only_on_weights=True,
    optimizer_params={'betas': [0.9, 0.96], 'eps': 1e-8, 'weight_decay': 1e-2},
    lr=5e-06,
    lr_scheduler='MultiStepLR',
    lr_scheduler_params={'milestones': [50000 * 18, 150000 * 18, 300000 * 18], 'gamma': 0.5, 'last_epoch': -1},
    test_sentences=[
        {
            'text': 'අයුබෝවන් මම සිංහලෙන් කතා කරනවා',
            'speaker_wav': SPEAKER_REFERENCE,
            'language': 'si',
        },
        {
            'text': 'ශ්‍රී ලංකාව ඉතා සුන්දර රටකි',
            'speaker_wav': SPEAKER_REFERENCE,
            'language': 'si',
        },
    ],
)

model = GPTTrainer.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    [config_dataset],
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

print(f'Train: {len(train_samples)}, Eval: {len(eval_samples)}')
print('\n🚀 Starting training...')

trainer = Trainer(
    TrainerArgs(
        restore_path=None,
        skip_train_epoch=False,
        start_with_eval=True,
        grad_accum_steps=GRAD_ACUMM_STEPS,
    ),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

## 5. Test the Fine-tuned Model

In [ ]:
# Find the best checkpoint
import glob
checkpoints = sorted(glob.glob('/content/xtts_sinhala/output/XTTS_Sinhala/XTTS_v2_Sinhala_FT-*/best_model*.pth'))
if not checkpoints:
    checkpoints = sorted(glob.glob('/content/xtts_sinhala/output/XTTS_Sinhala/XTTS_v2_Sinhala_FT-*/*.pth'))
print(f'Found checkpoints: {checkpoints}')

# Get the run directory
run_dirs = sorted(glob.glob('/content/xtts_sinhala/output/XTTS_Sinhala/XTTS_v2_Sinhala_FT-*/'))
if run_dirs:
    RUN_DIR = run_dirs[-1]
    print(f'Latest run: {RUN_DIR}')
    !ls -lh {RUN_DIR}

In [ ]:
# Test inference with the fine-tuned model
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import torch, torchaudio

# Load model from the latest checkpoint
config = XttsConfig()
config.load_json(os.path.join(RUN_DIR, 'config.json'))
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=RUN_DIR, eval=True)
model.cuda()

print('✅ Model loaded')

# Test sentences
test_sentences = [
    ('අයුබෝවන් මම සිංහලෙන් කතා කරනවා', 'test1_ayubowan.wav'),
    ('මේ අපේ රටයි', 'test2_rate.wav'),
    ('ඔබට සුභ දවසක් වේවා', 'test3_subha.wav'),
    ('ශ්‍රී ලංකාව ඉතා සුන්දර රටකි', 'test4_lanka.wav'),
    ('මම වෛද්‍යවරයෙකු හමුවීමට අවශ්‍යයි', 'test5_medical.wav'),
]

os.makedirs('/content/xtts_sinhala/test_output', exist_ok=True)

gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
    audio_path=['/content/xtts_sinhala/reference/ref_speaker.wav']
)

for text, filename in test_sentences:
    print(f'Generating: {text}')
    out = model.inference(
        text=text,
        language='si',
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.7,
    )
    wav = torch.tensor(out['wav']).unsqueeze(0)
    torchaudio.save(f'/content/xtts_sinhala/test_output/{filename}', wav, 24000)
    print(f'  ✅ Saved {filename}')

print('\n🎤 All test files generated!')
!ls -lh /content/xtts_sinhala/test_output/

In [ ]:
# Play the test audio in Colab
from IPython.display import Audio, display

for text, filename in test_sentences:
    print(f'\n🔊 {text}')
    display(Audio(f'/content/xtts_sinhala/test_output/{filename}'))

## 6. Voice Cloning Test
Upload your own voice clip (WAV, 6+ seconds) and clone it speaking Sinhala!

In [ ]:
# Upload a voice clip for cloning
from google.colab import files
uploaded = files.upload()

clone_wav = list(uploaded.keys())[0]
clone_path = f'/content/xtts_sinhala/{clone_wav}'
with open(clone_path, 'wb') as f:
    f.write(uploaded[clone_wav])

print(f'\n✅ Uploaded: {clone_wav}')

# Clone voice
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
    audio_path=[clone_path]
)

clone_text = 'අයුබෝවන් මම ඔබේ හඬින් සිංහලෙන් කතා කරනවා'
print(f'Generating with cloned voice: {clone_text}')

out = model.inference(
    text=clone_text,
    language='si',
    gpt_cond_latent=gpt_cond_latent,
    speaker_embedding=speaker_embedding,
    temperature=0.7,
)

wav = torch.tensor(out['wav']).unsqueeze(0)
torchaudio.save('/content/xtts_sinhala/test_output/voice_clone.wav', wav, 24000)

print('🎤 Cloned voice output:')
display(Audio('/content/xtts_sinhala/test_output/voice_clone.wav'))

## 7. Save Model to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = '/content/drive/MyDrive/xtts-sinhala-model'
os.makedirs(save_dir, exist_ok=True)

# Copy the fine-tuned model files
for f in os.listdir(RUN_DIR):
    src = os.path.join(RUN_DIR, f)
    if os.path.isfile(src):
        print(f'Copying {f}...')
        shutil.copy2(src, os.path.join(save_dir, f))

# Also copy test outputs
shutil.copytree('/content/xtts_sinhala/test_output', os.path.join(save_dir, 'test_output'), dirs_exist_ok=True)

print(f'\n✅ Model saved to Google Drive: {save_dir}')
!ls -lh {save_dir}